In [4]:
import os
import json
import random
import numpy as np
from pathlib import Path

# 设置随机种子
random.seed(42)
np.random.seed(42)

# 生成34个文件编号：1-11, 13-35（排除12）
files = list(range(1, 12)) + list(range(13, 36))
print(f"总文件数量: {len(files)} 个文件")
print(f"文件列表: {files}\n")

# 创建输出目录
output_dir = Path(r"D:\Project_Github\audio_click_mil\processed_data\folds")
output_dir.mkdir(parents=True, exist_ok=True)

# 随机打乱文件
shuffled_files = files[:]
random.shuffle(shuffled_files)

# 分成10个fold（标准10-fold CV）
n_folds = 10
fold_size = len(shuffled_files) // n_folds
folds = []

for i in range(n_folds):
    start = i * fold_size
    end = start + fold_size if i < n_folds - 1 else len(shuffled_files)
    test_files = sorted(shuffled_files[start:end])
    folds.append(test_files)

# 为每一折生成train/val/test配置并保存JSON
fold_configs = []
for fold_idx, test_files in enumerate(folds):
    # 剩余文件 = 所有文件 - 当前test_files
    remaining_files = sorted(set(files) - set(test_files))
    
    # 按80/20划分train/val（严格数量，取整）
    n_remaining = len(remaining_files)
    n_train = int(0.8 * n_remaining)
    n_val = n_remaining - n_train
    
    # 随机打乱剩余文件后划分（使用相同种子保证可重复）
    random.shuffle(remaining_files)
    train_files = sorted(remaining_files[:n_train])
    val_files = sorted(remaining_files[n_train:])
    
    config = {
        "fold": fold_idx,
        "test_files": test_files,
        "train_files": train_files,
        "val_files": val_files
    }
    
    fold_configs.append(config)
    
    # 保存单个fold的json
    json_path = output_dir / f"fold_{fold_idx}.json"
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(config, f, indent=2, ensure_ascii=False)
    
    print(f"Fold {fold_idx}:")
    print(f"  Test  : {len(test_files)} files → {test_files}")
    print(f"  Train : {len(train_files)} files → {train_files}")
    print(f"  Val   : {len(val_files)} files → {val_files}")
    print(f"  已保存 → {json_path}\n")

# 保存所有fold的汇总信息（可选）
summary_path = output_dir / "folds_summary.json"
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump({
        "total_files": len(files),
        "n_folds": n_folds,
        "folds": fold_configs
    }, f, indent=2, ensure_ascii=False)

print(f"所有10折划分完成！")
print(f"JSON文件保存在: {output_dir}")
print(f"汇总文件: {summary_path}")

总文件数量: 34 个文件
文件列表: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35]

Fold 0:
  Test  : 3 files → [6, 21, 22]
  Train : 24 files → [1, 2, 3, 7, 9, 11, 14, 15, 16, 17, 18, 19, 20, 23, 24, 25, 26, 27, 28, 30, 31, 32, 33, 34]
  Val   : 7 files → [4, 5, 8, 10, 13, 29, 35]
  已保存 → D:\Project_Github\audio_click_mil\processed_data\folds\fold_0.json

Fold 1:
  Test  : 3 files → [14, 26, 30]
  Train : 24 files → [1, 2, 3, 5, 6, 7, 10, 11, 13, 16, 18, 19, 20, 21, 22, 23, 24, 25, 27, 28, 29, 31, 33, 34]
  Val   : 7 files → [4, 8, 9, 15, 17, 32, 35]
  已保存 → D:\Project_Github\audio_click_mil\processed_data\folds\fold_1.json

Fold 2:
  Test  : 3 files → [11, 17, 18]
  Train : 24 files → [1, 2, 4, 5, 6, 8, 10, 14, 16, 19, 20, 21, 22, 24, 25, 26, 28, 29, 30, 31, 32, 33, 34, 35]
  Val   : 7 files → [3, 7, 9, 13, 15, 23, 27]
  已保存 → D:\Project_Github\audio_click_mil\processed_data\folds\fold_2.json

Fold 3:
  Test  : 3 files 